# E05 — walk-forward check of the E03 configuration (train + validation data only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about 60–75 minutes. Keep the tab open.
If Colab disconnects, just Run all again: finished folds are skipped.

**Question (fixed before any run, experiment ledger E05):** is the main model's weak validation score specific to the
2024 validation period (a change in market behaviour), or is it weak in every period?

**Design:** 4 expanding windows. Each fold trains on everything before its scored window (minus a 10-trading-day purge),
with labels (k, volatility scale), the input scaler and early stopping all refitted on that fold's training rows only:

| Fold | Scored window |
|---|---|
| 2022H2 | 2022-07-01 → 2022-12-31 |
| 2023H1 | 2023-01-01 → 2023-06-30 |
| 2023H2 | 2023-07-01 → 2023-12-31 |
| 2024H1 | 2024-01-01 → 2024-06-30 (overlaps the official validation block) |

Model = E03 configuration exactly (Multi-Scale PatchTST + sector + news + E03 inputs; lr 1e-4, dropout 0.2, max 30,
patience 10; seeds 20261015/16/17). XGBoost with the same 29 price inputs and the one-rule floors are recomputed per fold.
The sealed test block (from 2025-01-23) is never trained on or scored; the split code refuses any window that would reach it.
Outputs in `MyDrive/Senior/results/e05/`.


In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os
for f in ['scripts/summarize_walk_forward.py', 'tests/test_walk_forward.py']:
    assert os.path.exists(f), f'stop: push {f} from your PC, then run again'
assert 'walk_forward_blocks' in open('src/splits.py', encoding='utf-8').read(), 'stop: push the latest src/splits.py'
assert '--eval-start' in open('scripts/baselines_deep.py', encoding='utf-8').read(), 'stop: push the latest baselines_deep.py'
for f in ['scripts/baselines_deep.py', 'scripts/baselines_tabular.py', 'src/splits.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
E05 = f'{OUT_DIR}/e05'
os.makedirs(E05, exist_ok=True)


In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Pipeline, look-ahead and walk-forward leak tests; the notebook stops here if any fails (about 5 minutes)
import subprocess
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py', 'tests/test_walk_forward.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-2500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed'


In [ ]:
# The 4 folds: main model (3 seeds, GPU) then XGBoost (CPU). Finished folds are skipped on a rerun.
import json, subprocess, sys
FOLDS = [('2022H2', '2022-07-01', '2022-12-31'), ('2023H1', '2023-01-01', '2023-06-30'),
         ('2023H2', '2023-07-01', '2023-12-31'), ('2024H1', '2024-01-01', '2024-06-30')]
def done(path, key):
    try:
        r = json.load(open(path))
        return len(r['models']['multiscale_sector']['runs']) == 3 if key == 'deep' else 'h10' in r['results']
    except Exception:
        return False
for label, es, ee in FOLDS:
    deep, xgb = f'{E05}/{label}_deep.json', f'{E05}/{label}_xgb.json'
    if not done(deep, 'deep'):
        print(f'=== {label} main model'); sys.stdout.flush()
        subprocess.run(['python', 'scripts/baselines_deep.py', '--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv',
                        '--extra-features', 'e03', '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10',
                        '--eval-start', es, '--eval-end', ee, '--out', deep], check=True)
    else:
        print(f'=== {label} main model: already done')
    if not done(xgb, 'xgb'):
        print(f'=== {label} XGBoost'); sys.stdout.flush()
        subprocess.run(['python', 'scripts/baselines_tabular.py', '--extra-features', 'e03', '--eval-start', es, '--eval-end', ee,
                        '--out', xgb], check=True)
    else:
        print(f'=== {label} XGBoost: already done')


In [ ]:
# Summary: per fold and horizon, main model vs XGBoost, floors and random, with the per-fold bar.
# Last row = the official split (E03 run and XGBoost + E03 inputs), for comparison.
!python scripts/summarize_walk_forward.py --model multiscale_sector \
    --fold 2022H2 "$OUT_DIR/e05/2022H2_deep.json" "$OUT_DIR/e05/2022H2_xgb.json" \
    --fold 2023H1 "$OUT_DIR/e05/2023H1_deep.json" "$OUT_DIR/e05/2023H1_xgb.json" \
    --fold 2023H2 "$OUT_DIR/e05/2023H2_deep.json" "$OUT_DIR/e05/2023H2_xgb.json" \
    --fold 2024H1 "$OUT_DIR/e05/2024H1_deep.json" "$OUT_DIR/e05/2024H1_xgb.json" \
    --fold official "$OUT_DIR/main_model_e03.json" "$OUT_DIR/xgb_e03.json" \
    --out "$OUT_DIR/e05/summary_e05.json"
